# Topic 16: Overfitting and Regularization: L2 Weight Decay, Dropout, and Early Stopping

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 16 lecture note](https://dongchul.kim/csci3320-fall2026/topic/15), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## Three splits, three jobs

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
X, y = load_breast_cancer(return_X_y=True)

def split3(X, y, seed=0):
    # 20% test first, then 25% of the rest (= 20% of all) for validation
    X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=seed)
    sc = StandardScaler().fit(X_tr)          # fit the scaler on the training rows only
    f = lambda a: torch.tensor(sc.transform(a), dtype=torch.float32)
    return f(X_tr), f(X_va), f(X_te), torch.tensor(y_tr), torch.tensor(y_va), torch.tensor(y_te)

X_tr, X_va, X_te, y_tr, y_va, y_te = split3(X, y)
print("train", tuple(X_tr.shape), " validation", tuple(X_va.shape), " test", tuple(X_te.shape))

## The model and the training loop

In [ ]:
def make_model(n_in, hidden, n_out, p_drop=0.0):
    layers, prev = [], n_in
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        if p_drop > 0:
            layers.append(nn.Dropout(p_drop))       # after each hidden ReLU, never after the output
        prev = h
    layers.append(nn.Linear(prev, n_out))
    return nn.Sequential(*layers)

print(make_model(30, [256, 128, 64], 2, p_drop=0.5))

In [ ]:
def evaluate(model, X, y, loss_fn):
    model.eval()                                    # dropout off
    with torch.no_grad():
        out = model(X)
        return loss_fn(out, y).item(), (out.argmax(1) == y).float().mean().item()


def train(model, data, epochs=200, lr=0.001, batch=32, weight_decay=0.0, patience=None, seed=0):
    X_tr, X_va, X_te, y_tr, y_va, y_te = data
    torch.manual_seed(seed)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val, best_state, best_epoch, waited = float("inf"), None, 0, 0
    for epoch in range(epochs):
        model.train()                               # dropout on
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        for split, Xs, ys in (("train", X_tr, y_tr), ("val", X_va, y_va)):
            l, a = evaluate(model, Xs, ys, loss_fn)
            hist[f"{split}_loss"].append(l)
            hist[f"{split}_acc"].append(a)
        if patience is not None:                    # early stopping (Tool 3)
            if hist["val_loss"][-1] < best_val:
                best_val, best_epoch, waited = hist["val_loss"][-1], epoch + 1, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                waited += 1
                if waited >= patience:
                    break
    if patience is not None:
        model.load_state_dict(best_state)           # go back to the best epoch
    test_loss, test_acc = evaluate(model, X_te, y_te, loss_fn)   # the test split, once
    return hist, {"test_loss": test_loss, "test_acc": test_acc,
                  "stopped_at": len(hist["val_loss"]), "best_epoch": best_epoch}

In [ ]:
def sq_weights(model):
    """How big are the weights? The sum of w squared over every layer."""
    return sum((p ** 2).sum().item() for n, p in model.named_parameters() if n.endswith("weight"))


def run(p_drop=0.0, weight_decay=0.0, patience=None, hidden=(256, 128, 64)):
    """Train the same design on three different splits (seeds 0, 1, 2) and average."""
    hists, results = [], []
    for seed in (0, 1, 2):
        data = split3(X, y, seed)
        torch.manual_seed(seed)
        model = make_model(30, list(hidden), 2, p_drop)
        h, r = train(model, data, weight_decay=weight_decay, patience=patience, seed=seed)
        r["sq_weights"] = sq_weights(model)
        r["final_train_loss"] = h["train_loss"][-1]
        hists.append(h)
        results.append(r)
    mean = {k: float(np.mean([r[k] for r in results]))
            for k in ("test_loss", "test_acc", "sq_weights", "final_train_loss")}
    return hists, results, mean

## The baseline: overfitting, measured

In [ ]:
base_hists, base_results, base = run()
val = np.mean([h["val_loss"] for h in base_hists], axis=0)      # mean validation curve
print(f"training loss at epoch 200:   {base['final_train_loss']:.4f}")
print(f"lowest validation loss:       {val.min():.3f} at epoch {val.argmin() + 1}")
print(f"validation loss at epoch 200: {val[-1]:.3f}")
print(f"test loss {base['test_loss']:.3f}   test accuracy {base['test_acc']:.3f}")

In [ ]:
def plot_curves(hists, title):
    tr = np.mean([h["train_loss"] for h in hists], axis=0)
    va = np.mean([h["val_loss"] for h in hists], axis=0)
    epochs = np.arange(1, len(tr) + 1)
    plt.figure(figsize=(6, 3.5))
    plt.plot(epochs, tr, label="train loss")
    plt.plot(epochs, va, label="validation loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.ylim(0, 0.45)
    plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.show()

plot_curves(base_hists, "Baseline (mean of 3 seeds)")

## Tool 1: L2 weight decay

In [ ]:
l2_hists, l2_results, l2 = run(weight_decay=1e-2)
print(f"            sum w^2   test loss   test acc")
print(f"no decay     {base['sq_weights']:6.1f}     {base['test_loss']:.3f}      {base['test_acc']:.3f}")
print(f"decay 1e-2   {l2['sq_weights']:6.1f}     {l2['test_loss']:.3f}      {l2['test_acc']:.3f}")

### How strong? Choose lambda on the validation set

In [ ]:
print("lambda    train loss   val loss   test acc")
for wd in [0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    hists, results, m = run(weight_decay=wd)
    val_end = np.mean([h["val_loss"][-1] for h in hists])
    print(f"{wd:<8}  {m['final_train_loss']:.4f}       {val_end:.3f}      {m['test_acc']:.3f}")

## Tool 2: dropout

In [ ]:
torch.manual_seed(0)
drop = nn.Dropout(p=0.5)
x = torch.ones(8)
drop.train(); print(drop(x))
drop.eval();  print(drop(x))

In [ ]:
do_hists, do_results, do = run(p_drop=0.5)
print(f"dropout 0.5: test loss {do['test_loss']:.3f}   test accuracy {do['test_acc']:.3f}")

In [ ]:
torch.manual_seed(0)
data = split3(X, y, 0)
model = make_model(30, [256, 128, 64], 2, p_drop=0.5)
train(model, data)
X_te, y_te = data[2], data[5]
accuracy = lambda out: (out.argmax(1) == y_te).float().mean().item()

model.train()                                   # the mistake: still in train mode
with torch.no_grad():
    print("train mode, 5 calls:", [round(accuracy(model(X_te)), 3) for _ in range(5)])
model.eval()                                    # the fix
with torch.no_grad():
    print("eval mode,  5 calls:", [round(accuracy(model(X_te)), 3) for _ in range(5)])

## Tool 3: early stopping

In [ ]:
es_hists, es_results, es = run(patience=10)
for seed, r in enumerate(es_results):
    print(f"seed {seed}: best epoch {r['best_epoch']:3d}, stopped at {r['stopped_at']:3d}, test loss {r['test_loss']:.3f}")
print(f"mean: test loss {es['test_loss']:.3f}   test accuracy {es['test_acc']:.3f}")

## Results side by side

In [ ]:
all_hists, all_results, both = run(p_drop=0.5, weight_decay=1e-2, patience=10)
import pandas as pd
table = pd.DataFrame([base, l2, do, es, both],
                     index=["Baseline", "L2", "Dropout", "Early stopping", "All three"])
print(table[["final_train_loss", "sq_weights", "test_loss", "test_acc"]].round(4))

Back to the [Topic 16 lecture note](https://dongchul.kim/csci3320-fall2026/topic/15).